In [4]:
!pip install scikit-image
!pip install open_clip_torch

In [14]:
import torch
from PIL import Image
import numpy as np
import os

from skimage.metrics import structural_similarity as ssim

import open_clip

In [15]:
def load_image(path):

    img = Image.open(path).convert("RGB")
    img = img.resize((512,512))
    
    return np.array(img)

In [16]:
def compute_ssim(img1, img2):

    score = ssim(
        img1,
        img2,
        channel_axis=2
    )

    return score

In [17]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

model = model.to(device)

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

D:\anaconda3\envs\diffusion\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\26911\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
D:\anaconda3\envs\diffusion\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch

In [18]:
def clip_score(image_path, text_prompt):

    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)

    text = open_clip.tokenize([text_prompt]).to(device)

    with torch.no_grad():

        image_features = model.encode_image(image)
        text_features = model.encode_text(text)

        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        score = (image_features @ text_features.T).item()

    return score

In [19]:
original = load_image("C:/Users/26911/Desktop/diffusion_style_project/data/content/content_001.jpg")
generated = load_image("C:/Users/26911/Desktop/diffusion_style_project/results/content_001_vangogh.png")

ssim_score = compute_ssim(original, generated)

print("SSIM:", ssim_score)

SSIM: 0.2664946999231737


In [20]:
clip = clip_score(
    "C:/Users/26911/Desktop/diffusion_style_project/results/content_001_vangogh.png",
    "a painting in the style of Vincent van Gogh"
)

print("CLIP Score:", clip)

CLIP Score: 0.27936139702796936


In [21]:
content_folder = "C:/Users/26911/Desktop/diffusion_style_project/data/content"
results_folder = "C:/Users/26911/Desktop/diffusion_style_project/results"

scores = []

for file in os.listdir(content_folder):

    name = file.split(".")[0]

    original = load_image(os.path.join(content_folder, file))

    stylized_path = os.path.join(results_folder, f"{name}_vangogh.png")

    if os.path.exists(stylized_path):

        stylized = load_image(stylized_path)

        s = compute_ssim(original, stylized)

        scores.append(s)

print("Average SSIM:", np.mean(scores))

Average SSIM: 0.32378842693437454
